# CARec training

- Load saved splits and cached item embeddings. Preparation stays in [data_exploration.ipynb](data_exploration.ipynb), encoding in [carec_encoder_comparison.ipynb](carec_encoder_comparison.ipynb).
- Phase 1 learns user vectors. Phase 2 trains the item MLP. Both use the training graph; cached text vectors stay fixed.
- Currently implemented: phase-one training and evaluation for one selected variant.


## 1. Settings and imports

- Select embeddings, device, and training parameters here.
- Default to MPS. Use `device_name = "cpu"` to switch back. Recreate the model after changing devices.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.data.dataset import Dataset
from recbole.utils import init_seed
from recbole.utils.enum_type import FeatureSource, FeatureType

carec = load_carec()

DATASET_ID = "electronics_carec_035_seed2023"
PREPARED_DIR = PROJECT_ROOT / "data/processed" / DATASET_ID
RECBOLE_ROOT = PROJECT_ROOT / "artifacts/recbole"
RECBOLE_DIR = RECBOLE_ROOT / DATASET_ID
EMBEDDING_NAME = "all-MiniLM-L6-v2.npz"
EMBEDDING_PATH = PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID / EMBEDDING_NAME

seed = 2023
phase1_epochs = 1
training_batch_size = 1024
learning_rate = 0.001

device_name = "mps"
if device_name == "mps" and not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable in this kernel. Select the project environment or use CPU.")
device = torch.device(device_name)


## 2. Load data with RecBole

- RecBole reads the saved splits and assigns integer IDs. It reserves ID 0 for padding.
- Export interaction files only when missing. All embedding variants use these same files.
- All splits share ID mappings. Only training interactions will enter the graph.


In [2]:
with np.load(EMBEDDING_PATH, allow_pickle=False) as saved:
    embedding_item_ids = saved["item_ids"]
    item_embeddings = saved["embeddings"]
    embedding_model_id = str(saved["model_id"])
    embedding_prompt = str(saved["prompt"])

RECBOLE_DIR.mkdir(parents=True, exist_ok=True)
interaction_columns = ["user_id:token", "item_id:token", "timestamp:float"]
for split_name in ("train", "valid", "test", "cold"):
    split_path = RECBOLE_DIR / f"{DATASET_ID}.{split_name}.inter"
    if not split_path.exists():
        interactions = pd.read_csv(
            PREPARED_DIR / f"{split_name}.tsv.gz",
            sep="\t",
            usecols=interaction_columns,
            dtype={"user_id:token": str, "item_id:token": str},
            float_precision="round_trip",
        )
        # RecBole requires typed headers. Leave source_row out of this export.
        interactions[interaction_columns].to_csv(split_path, sep="\t", index=False)

catalogue_path = RECBOLE_DIR / f"{DATASET_ID}.item"
if not catalogue_path.exists():
    pd.DataFrame({"item_id:token": embedding_item_ids}).to_csv(
        catalogue_path, sep="\t", index=False,
    )

print(f"{embedding_model_id}: {item_embeddings.shape}, prompt={embedding_prompt!r}")


sentence-transformers/all-MiniLM-L6-v2: (34220, 384), prompt=''


In [ ]:
dataset_settings = {
    "data_path": str(RECBOLE_ROOT),
    "benchmark_filename": ["train", "valid", "test", "cold"],
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"],
        "item": ["item_id"],
    },
    "normalize_all": False,
    "use_gpu": False,  # RecBole uses this flag for CUDA, not MPS.
}

config = Config(model=carec.CollabContex, dataset=DATASET_ID, config_dict=dataset_settings)
# Set before construction: CARec stores graph and item tensors outside registered buffers.
config["device"] = device
recbole_dataset = Dataset(config)
recbole_product_ids = recbole_dataset.field2id_token["item_id"]


## 3. Attach embeddings and build splits

- Cache rows and RecBole IDs can have different orders. Match them by original product ID and leave padding as zeros.
- Register the vectors as `item_emb`, the feature expected by CARec.
- `build()` converts tables to tensors and separates the saved splits. Rerun section 2 before repeating this section.


In [4]:
embedding_row_by_product = {}
for embedding_row, product_id in enumerate(embedding_item_ids):
    embedding_row_by_product[product_id] = embedding_row

assert len(embedding_row_by_product) == len(embedding_item_ids), "Duplicate product IDs in cache"
assert recbole_product_ids[0] == "[PAD]"
assert set(recbole_product_ids[1:]) == set(embedding_item_ids), "Product catalogues differ"

embedding_dimension = item_embeddings.shape[1]
recbole_item_embeddings = np.zeros(
    (recbole_dataset.item_num, embedding_dimension), dtype=np.float32,
)
# Row number must equal the RecBole product ID, not the original cache row.
for recbole_id in range(1, recbole_dataset.item_num):
    product_id = recbole_product_ids[recbole_id]
    recbole_item_embeddings[recbole_id] = item_embeddings[embedding_row_by_product[product_id]]

recbole_dataset.set_field_property(
    "item_emb", FeatureType.FLOAT_SEQ, FeatureSource.ITEM, embedding_dimension,
)
recbole_dataset.item_feat["item_emb"] = list(recbole_item_embeddings)
train_dataset, valid_dataset, test_dataset, cold_dataset = recbole_dataset.build()


## 4. Create a fresh CARec model

- Use only training edges. Match model dimensions to the selected embeddings.
- Rerun this section to reset weights before a new run.
- On MPS, uniformity runs on CPU with gradients preserved. Graph propagation and model parameters stay on MPS.


In [ ]:
config["seed"] = seed
config["encoder"] = "LightGCN"
config["num_layers"] = 2
config["embedding_size"] = embedding_dimension
config["mlp"] = {
    "input_dim": embedding_dimension,
    "hidden_dims": [embedding_dimension, embedding_dimension],
    "output_dim": embedding_dimension,
    "num_layers": 2,
    "dropout": 0.2,
}

init_seed(config["seed"], config["reproducibility"])
carec_model = carec.CollabContex(config, train_dataset).to(config["device"])

if device.type == "mps":
    # pdist lacks an MPS kernel. CPU transfer preserves autograd and the original loss.
    def mps_uniformity(vectors, t=2):
        return carec.CollabContex.uniformity(vectors.cpu(), t=t).to(vectors.device)

    carec_model.uniformity = mps_uniformity

assert carec_model.encoder.user_embedding.weight.device.type == device.type
assert carec_model.encoder.item_embedding.device.type == device.type
assert carec_model.encoder.edge_index.device.type == device.type
assert carec_model.encoder.edge_weight.device.type == device.type

print(f"Embedding variant: {EMBEDDING_NAME}, dimension: {embedding_dimension}")
print(f"Users: {train_dataset.user_num - 1:,}, products: {train_dataset.item_num - 1:,}")
print({name: len(split.inter_feat) for name, split in zip(
    config["benchmark_filename"], [train_dataset, valid_dataset, test_dataset, cold_dataset],
)})
print(f"Directed graph entries: {carec_model.encoder.edge_index.shape[1]:,}")
print(f"Device: {config['device']}")


## 5. Train phase 1

- Shuffle interactions and update user vectors. Every batch propagates over the full training graph.
- Loss combines alignment and user uniformity; negative values are possible.
- Rerunning continues current weights with a fresh optimizer. For a fresh start, rerun section 4 first.


In [6]:
carec_model.set_item_tutoring_phase()
carec_model.train()
# Phase 1 bypasses the MLP. Only the user table is optimized.
optimizer = torch.optim.Adam(
    carec_model.encoder.user_embedding.parameters(),
    lr=learning_rate,
)
training_generator = torch.Generator().manual_seed(config["seed"])
phase1_losses = []


In [7]:
for epoch in range(phase1_epochs):
    shuffled_indices = torch.randperm(
        len(train_dataset.inter_feat), generator=training_generator
    )
    total_loss = 0.0
    interaction_count = 0
    batches = tqdm(
        shuffled_indices.split(training_batch_size),
        desc=f"Phase 1, epoch {epoch + 1}/{phase1_epochs}",
    )

    for indices in batches:
        interaction_batch = train_dataset.inter_feat[indices].to(config["device"])
        if interaction_batch[config["USER_ID_FIELD"]].unique().numel() < 2:
            raise ValueError("User uniformity needs at least two distinct users.")

        optimizer.zero_grad(set_to_none=True)
        loss = carec_model.calculate_loss(interaction_batch)
        if not torch.isfinite(loss):
            raise ValueError("Training loss is not finite.")
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        total_loss += batch_loss * len(interaction_batch)
        interaction_count += len(interaction_batch)
        batches.set_postfix(loss=f"{batch_loss:.4f}")

    average_loss = total_loss / interaction_count
    phase1_losses.append(average_loss)
    print(f"Epoch {epoch + 1}: average batch loss = {average_loss:.6f}")

carec_model.eval()


Phase 1, epoch 1/1: 100%|██████████| 442/442 [06:33<00:00,  1.12it/s, loss=-2.5687]

Epoch 1: average batch loss = -2.568197


CollabContex(
  (encoder): LGCNEncoder(
    (gcn_conv): LightGCNConv(384)
    (mlp): MLP(
      (mlp): Sequential(
        (0): Linear(in_features=384, out_features=384, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.2, inplace=False)
        (3): Linear(in_features=384, out_features=384, bias=True)
        (4): ReLU()
        (5): Dropout(p=0.2, inplace=False)
        (6): Linear(in_features=384, out_features=384, bias=True)
      )
    )
    (user_embedding): Embedding(81521, 384)
  )
)

## 6. Evaluate current weights

- RecBole computes per-user Recall/NDCG at 10 and 50. Warm ranks the full catalogue; cold ranks cold products only and bypasses MLP. [Paper, section 4.1.3](https://yangliangwei.github.io/publication/cikm2024_carec/CIKM2024_CARec.pdf).
- Mask padding and earlier history, except current positives. Repeated pairs across splits remain; these are pilot results.
- Run after section 5. Recreating the model in section 4 resets trained weights. Use validation for later checkpoint selection.


In [8]:
from recbole.data.dataloader import FullSortEvalDataLoader
from recbole.sampler import Sampler
from recbole.evaluator import Collector, Evaluator

config["metrics"] = ["Recall", "NDCG"]
config["topk"] = [10, 50]
# RecBole measures eval_batch_size in user-product scores, not users.
config["eval_batch_size"] = 64 * train_dataset.item_num

# Upstream repeatable sampling leaves history empty. Supply cumulative history.
history_sampler = Sampler(
    ["train", "valid", "test", "cold"],
    [train_dataset, valid_dataset, test_dataset, cold_dataset],
)
evaluation_loaders = {}
for split_name, split in [("valid", valid_dataset), ("warm", test_dataset), ("cold", cold_dataset)]:
    phase = "test" if split_name == "warm" else split_name
    evaluation_loaders[split_name] = FullSortEvalDataLoader(
        config, split, history_sampler.set_phase(phase), shuffle=False,
    )

cold_product_ids = cold_dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
train_product_ids = train_dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
assert not set(cold_product_ids.tolist()) & set(train_product_ids.tolist())
print({name: len(loader.uid_list) for name, loader in evaluation_loaders.items()})


{'valid': 12386, 'warm': 81520, 'cold': 18680}


In [ ]:
@torch.no_grad()
def evaluate_carec(model, loader, cold_only=False):
    collector = Collector(config)
    item_count = loader.dataset.item_num
    evaluator = Evaluator(config)
    previous_training = model.training
    previous_mlp_bypass = model.encoder.not_apply_mlp
    model.eval()
    if cold_only:
        model.encoder.not_apply_mlp = True
    # full_sort_predict caches graph outputs. Never reuse them across modes or epochs.
    model.restore_user_e = model.restore_item_e = None
    candidate_mask = torch.ones(item_count, dtype=torch.bool, device=config["device"])
    if cold_only:
        candidate_mask[:] = False
        cold_ids = loader.dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
        candidate_mask[cold_ids.to(config["device"])] = True
    candidate_mask[0] = False

    try:
        for interaction, history, positive_users, positive_items in tqdm(loader, desc="Cold" if cold_only else "Full ranking"):
            scores = model.full_sort_predict(interaction.to(config["device"]))
            scores = scores.view(-1, item_count)
            if not torch.isfinite(scores).all():
                raise ValueError("Non-finite recommendation scores before masking.")
            # Apply the item mask to columns, for every user in the batch.
            scores[:, ~candidate_mask] = -torch.inf
            if history is not None:
                scores[history[0].to(scores.device), history[1].to(scores.device)] = -torch.inf
            collector.eval_batch_collect(scores, interaction, positive_users, positive_items)
        return dict(evaluator.evaluate(collector.get_data_struct()))
    finally:
        model.encoder.not_apply_mlp = previous_mlp_bypass
        model.restore_user_e = model.restore_item_e = None
        model.train(previous_training)


In [11]:
if not phase1_losses:
    raise ValueError("Run phase-one training before evaluating this pilot.")
if not carec_model.encoder.not_apply_mlp:
    raise ValueError("This pilot expects phase-one weights with MLP bypassed.")

pilot_results = {}
for split_name, loader in evaluation_loaders.items():
    pilot_results[split_name] = evaluate_carec(
        carec_model, loader, cold_only=(split_name == "cold"),
    )
display(pd.DataFrame(pilot_results).T)


Cold: 100%|██████████| 292/292 [00:02<00:00, 132.24it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
valid,0.0464,0.0941,0.0295,0.0401
warm,0.0666,0.1098,0.0515,0.0622
cold,0.0679,0.1515,0.0411,0.0600
